# DATA 622: Homework 8
**Author**: Brett Allen (ballen3@umbc.edu)

**Date Completed:** 03/29/2026

## Setup

In [1]:
!python -m pip install -q requests beautifulsoup4 transformers sentence-transformers nltk scikit-learn textblob "lxml[html_clean]"

### Imports

In [2]:
import requests
from bs4 import BeautifulSoup
import nltk
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from collections import Counter
import re
import json

/home/brett/anaconda3/envs/data-science/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/home/brett/anaconda3/envs/data-science/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configurations

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /home/brett/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/brett/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/brett/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/115.0"
}

## Questions
Use these two articles:
* "https://www.nytimes.com/2024/05/14/climate/climate-change-extreme-weather.html"
* "https://www.foxnews.com/science/climate-change-weather-impact-explained"

In [11]:
def fetch_article_text(url, timeout=None):
    response = requests.get(url, headers=headers, timeout=timeout)
    soup = BeautifulSoup(response.text, "html.parser")

    # Extract all paragraph text
    paragraphs = soup.find_all("p")
    text = " ".join([p.get_text() for p in paragraphs])

    return text

In [6]:
url1 = "https://www.nytimes.com/2024/05/14/climate/climate-change-extreme-weather.html"
url2 = "https://www.foxnews.com/science/climate-change-weather-impact-explained"

In [7]:
text1 = fetch_article_text(url1)
text2 = fetch_article_text(url2)

print("Article 1 length:", len(text1))
print("Article 2 length:", len(text2))

Article 1 length: 43
Article 2 length: 783


In [8]:
print(text1)

Please enable JS and disable any ad blocker


In [9]:
print(text2)


        This material may not be published, broadcast, rewritten, or redistributed. ©2026 FOX News Network, LLC. All rights reserved. Quotes displayed in real-time or delayed by at least 15 minutes. Market data provided by Factset. Powered and implemented by FactSet Digital Solutions. Legal Statement. Mutual Fund and ETF data provided by LSEG.
       It seems you've stumbled upon our 404 page. 
        This material may not be published, broadcast, rewritten, or redistributed. ©2026 FOX News Network, LLC. All rights reserved. Quotes displayed in real-time or delayed by at least 15 minutes. Market data provided by
        Factset. Powered and implemented by
        FactSet Digital Solutions.
        Legal Statement. Mutual Fund and ETF data provided by
        LSEG.
      


**Observation**: Unfortunately, neither of the articles are accessible via web scraping. Will need to find alternative related articles.

In [12]:
url1 = "https://wapo.st/4bEXxGl"
text1 = fetch_article_text(url1, timeout=5)
print("Article 1 length:", len(text1))

ReadTimeout: HTTPSConnectionPool(host='www.washingtonpost.com', port=443): Read timed out. (read timeout=5)

**Observation:** The alternate URL provided by the professor doesn't work either (timeout); likely because there is a paywall for washington post articles.

In [13]:
url1 = "https://earthjustice.org/feature/how-climate-change-is-fueling-extreme-weather"
text1 = fetch_article_text(url1)
print("Article 1 length:", len(text1))

Article 1 length: 9416


In [14]:
# Print first 1000 characters
print(text1[:1000] + '...')

We take on many of the biggest environmental and health challenges of our time and stick with them. The law makes change. Because the earth needs a good lawyer. We take on many of the biggest environmental and health challenges of our time and stick with them. The law makes change. Because the earth needs a good lawyer. our Stories July 28, 2025 Carbon pollution is contributing to climate disasters that will only get worse unless we take action. Carbon pollution is contributing to climate disasters that will only get worse unless we take action. Across the globe, extreme weather is becoming the new normal. From season to season and year to year, weather events that were once rare occurrences are now increasingly commonplace. Human activity is causing rapid changes to our global climate that are contributing to extreme weather conditions. When fossil fuels are burned for electricity, heat, and transportation, carbon dioxide, a greenhouse gas that traps solar radiation, is released into 

In [15]:
url2 = "https://interactive.carbonbrief.org/attribution-studies/index.html"
text2 = fetch_article_text(url2)
print("Article 2 length:", len(text2))

Article 2 length: 16662


In [16]:
# Print first 1000 characters
print(text2[:1000] + '...')

  
				18 November 2024Last updated: 19 March 2026 By
			Robert McSweeney and Ayesha Tandon  Design and development by
			Kerry Cleaver,
					 Tom Pearson and Tom Prater  In 2004, a trio of researchers published a study that accomplished something never seen before. They calculated the specific contribution that human-caused climate change made to an individual extreme weather event. The extreme event in question was the European heatwave in the summer of 2003. Week upon week of extreme heat had a devastating impact, killing more than 70,000 people across the continent. The scientists worked out that human influence had at least doubled the risk of such an extreme heatwave occurring. The findings made headlines around the world. The study kick-started the scientific field of “extreme event attribution”. Attribution studies calculate whether, and by how much, climate change affected the intensity, frequency or impact of extremes – from wildfires in the US and drought in South Africa thr

### 1. Based on AI/ML methods, measure the similarity between the two articles.

In [17]:
# Load sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1477.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
# Create embeddings for the article texts (tokenizing) to prepare for cosine similarity comparison
embeddings = model.encode([text1, text2])

In [19]:
similarity_score = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
print(f"Semantic Similarity Score: {similarity_score:.4f}")

Semantic Similarity Score: 0.6430


### 2. Determine if they share...

#### a. Similar Topics

In [20]:
def extract_keywords(text, top_n=10):
    # Identify all words
    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())

    # Remove stopwords
    stopwords = set(nltk.corpus.stopwords.words('english'))
    words = [w for w in words if w not in stopwords]

    # Return top N words
    return [word for word, _ in Counter(words).most_common(top_n)]

In [21]:
keywords1 = extract_keywords(text1)
keywords2 = extract_keywords(text2)

print("Article 1 Keywords:", json.dumps(keywords1, indent=2))
print("Article 2 Keywords:", json.dumps(keywords2, indent=2))

Article 1 Keywords: [
  "temperatures",
  "climate",
  "global",
  "change",
  "energy",
  "extreme",
  "water",
  "california",
  "take",
  "pollution"
]
Article 2 Keywords: [
  "climate",
  "studies",
  "attribution",
  "likely",
  "change",
  "severe",
  "extremes",
  "heat",
  "human",
  "extreme"
]


In [22]:
# Determine shared topics based on overlapping keywords between the articles
common_topics = set(keywords1).intersection(set(keywords2))
print("Shared Keywords (Topic Overlap):", json.dumps(common_topics, indent=2, default=str))

Shared Keywords (Topic Overlap): "{'climate', 'extreme', 'change'}"


#### b. Sentiment

In [40]:
# Create sentiment analysis pipeline with transformers
sentiment_pipeline = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 2009.34it/s]


In [42]:
# Need to limit due to token size
# TODO (Future work): Determine if there's a better model that has a context window large enough for both articles or use chunking
article1_sentiment = sentiment_pipeline(text1[:512])[0]
article2_sentiment = sentiment_pipeline(text2[:512])[0]

print("Article 1 Sentiment:", article1_sentiment)
print("Article 2 Sentiment:", article2_sentiment)

Article 1 Sentiment: {'label': 'POSITIVE', 'score': 0.9129547476768494}
Article 2 Sentiment: {'label': 'NEGATIVE', 'score': 0.9743443727493286}


#### c. Emotions

In [43]:
# Create a text classification pipeline using an emotion detection transformer model
emotion_pipeline = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=1
)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 69496.91it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
emotion1 = emotion_pipeline(text1[:512])[0][0]
emotion2 = emotion_pipeline(text2[:512])[0][0]

print("Article 1 Emotion:", emotion1)
print("Article 2 Emotion:", emotion2)

Article 1 Emotion: {'label': 'neutral', 'score': 0.587785005569458}
Article 2 Emotion: {'label': 'fear', 'score': 0.331584095954895}


### 3. Identify the top five keywords in each article. 

In [46]:
article1_top5_keywords = extract_keywords(text1, top_n=5)
article2_top5_keywords = extract_keywords(text2, top_n=5)

print("Top 5 Keywords from Article 1:", json.dumps(article1_top5_keywords, indent=2))
print("Top 5 Keywords from Article 2:", json.dumps(article2_top5_keywords, indent=2))

Top 5 Keywords from Article 1: [
  "temperatures",
  "climate",
  "global",
  "change",
  "energy"
]
Top 5 Keywords from Article 2: [
  "climate",
  "studies",
  "attribution",
  "likely",
  "change"
]


### 4. Summarize your findings using LLMs.

In [47]:
# Initialize the LLM
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 338/338 [00:01<00:00, 228.68it/s]


In [50]:
prompt = f"""
Analyze the following comparison between two climate articles:

Similarity Score: {similarity_score}

Article 1 Keywords: {article1_top5_keywords}
Article 2 Keywords: {article2_top5_keywords}

Article 1 Sentiment: {article1_sentiment}
Article 2 Sentiment: {article2_sentiment}

Article 1 Emotion: {emotion1}
Article 2 Emotion: {emotion2}

Provide a concise summary of:
1. Topic similarity
2. Differences in tone/sentiment
3. Key insights

Only provide the summary.
"""

result = llm(
    prompt,
    max_length=512,
    temperature=0.3
)

summary = result[0]["generated_text"]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [52]:
# Remove prompt from the summary if it exists
if prompt in summary:
    idx = summary.index(prompt)
    summary = summary[idx+len(prompt):]
else:
    print('Prompt is not in the summary.')

In [54]:
print("="*60)
print("LLM Summary of Findings:")
print("="*60)
print(summary)

LLM Summary of Findings:
### Summary:

#### Topic Similarity:
The topics covered by the two articles are somewhat similar, as both discuss aspects related to climate change and its impacts on global temperatures.

#### Differences in Tone/Sentiment:
- **Tone:** Article 1 has a positive sentiment, indicating an optimistic view or understanding of climate change and its effects. 
- **Sentiment:** Article 2 has a negative sentiment, suggesting concern or fear about climate change.

#### Key Insights:
- **Article 1** discusses various aspects of climate change, including global temperature changes and energy usage, with a generally positive outlook on these issues.
- **Article 2** focuses more on studies attributing climate change, likely implying that it is a subject of study rather than one's personal experience or immediate concerns. It also mentions "likely change," which could be interpreted as a cautious or uncertain stance towards the severity of climate change. 

These differences 